# Step 10 · Inspect native V4 Explorer coordinates

The Explorer uses the current native V4 run. Coordinates remain empty until a native V4 embedding is fitted; this notebook checks coverage and renders a plot only when coordinates are available.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT, LOG, LOG_PATH = configure_notebook('10_inspect_embeddings', 'explorer-inspection')
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 21:50 INFO Notebook session started at 2026-11-02T21:50:45.375648+00:00


2026-11-02 21:50 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 21:50 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/explorer-inspection/10_inspect_embeddings.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/explorer-inspection/10_inspect_embeddings.log


## Step 1 · Load the current Explorer profile payload

In [2]:
import gzip, json, pandas as pd
from IPython.display import display
PAYLOAD_PATH = ROOT / 'sites/pathogen-atlas-explorer/explorer-data.json.gz'
payload = json.loads(gzip.open(PAYLOAD_PATH, 'rt', encoding='utf-8').read())
profiles = pd.DataFrame(payload['species'])
print('Payload:', PAYLOAD_PATH)
print('Groups:', payload['groups'], '| profiles:', len(profiles), '| coordinates_ready:', payload['coordinates_ready'])
display(profiles.groupby('organism_group').agg(profiles=('organism_id','nunique'),coordinates=('x','count')))
LOG.info('Loaded Explorer payload profiles=%d groups=%s', len(profiles), payload['groups'])

Payload: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/sites/pathogen-atlas-explorer/explorer-data.json.gz
Groups: ['viruses'] | profiles: 17554 | coordinates_ready: False


,profiles,coordinates
organism_group,,
viruses,17554,0


2026-11-02 21:50 INFO Loaded Explorer payload profiles=17554 groups=['viruses']


## Step 2 · Select one group and check coordinate completeness

In [3]:
GROUP_TO_PLOT = 'viruses'
group = profiles.loc[profiles.organism_group.eq(GROUP_TO_PLOT)].copy()
group['x'] = pd.to_numeric(group.x, errors='coerce')
group['y'] = pd.to_numeric(group.y, errors='coerce')
plot_data = group.dropna(subset=['x','y'])
print('Selected group:', GROUP_TO_PLOT, '| profiles:', len(group), '| drawable:', len(plot_data))
if plot_data.empty: print('No fitted coordinates are available for this group yet.')
LOG.info('Coordinate check group=%s drawable=%d missing=%d', GROUP_TO_PLOT, len(plot_data), len(group)-len(plot_data))


2026-11-02 21:50 INFO Coordinate check group=viruses drawable=0 missing=17554


Selected group: viruses | profiles: 17554 | drawable: 0
No fitted coordinates are available for this group yet.


## Step 3 · Render a labeled SVG scatter plot

In [4]:
from html import escape
if len(plot_data):
    width, height, pad = 900, 560, 42
    x0, x1 = plot_data.x.min(), plot_data.x.max(); y0, y1 = plot_data.y.min(), plot_data.y.max()
    if x0 == x1: x1 = x0 + 1
    if y0 == y1: y1 = y0 + 1
    marks = []
    for row in plot_data.itertuples(index=False):
        x = pad + (row.x-x0)/(x1-x0)*(width-2*pad)
        y = height-pad-(row.y-y0)/(y1-y0)*(height-2*pad)
        title = escape(str(row.accepted_name))
        marks.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3" fill="#21857b" fill-opacity=".55"><title>{title}</title></circle>')
    svg = (f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">' f'<rect width="100%" height="100%" fill="white"/>{"".join(marks)}</svg>')
    from IPython.display import SVG, display
    display(SVG(svg))
else:
    svg = None
LOG.info('Rendered %d profile coordinates for %s', len(plot_data), GROUP_TO_PLOT)


2026-11-02 21:50 INFO Rendered 0 profile coordinates for viruses


## Step 4 · Record the plotted release and export

In [5]:
plot_record = {'source_payload': str(PAYLOAD_PATH.relative_to(ROOT)),
               'source_release': payload.get('schema_version'), 'group': GROUP_TO_PLOT,
               'profiles_plotted': len(plot_data), 'coordinates_missing': len(group)-len(plot_data),
               'run_id': payload.get('stats', {}).get('run_id')}
PLOT_DIR = ROOT / 'data/v4/plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
if svg is not None:
    SVG_PATH = PLOT_DIR / f'{GROUP_TO_PLOT}-{payload.get("schema_version","explorer")}.svg'
    SVG_PATH.write_text(svg, encoding='utf-8')
    print('SVG saved:', SVG_PATH)
else:
    print('SVG export skipped because this run has no fitted coordinates.')
display(plot_record)
LOG.info('Coordinate inspection complete: %s', plot_record)


SVG export skipped because this run has no fitted coordinates.


{'source_payload': 'sites/pathogen-atlas-explorer/explorer-data.json.gz',
 'source_release': 'v4-native-pair-screen-v1',
 'group': 'viruses',
 'profiles_plotted': 0,
 'coordinates_missing': 17554,
 'run_id': 'pilot-v1'}

2026-11-02 21:50 INFO Coordinate inspection complete: {'source_payload': 'sites/pathogen-atlas-explorer/explorer-data.json.gz', 'source_release': 'v4-native-pair-screen-v1', 'group': 'viruses', 'profiles_plotted': 0, 'coordinates_missing': 17554, 'run_id': 'pilot-v1'}


## Step 5 · Native embedding handoff

In [6]:
EMBEDDING_RUN_READY = False
print('Native V4 embedding fitting is a separate pipeline stage. Record the resolved feature-profile hash, selected features, seed, software versions, and input run before fitting.')
LOG.info('Native embedding fit not started; baseline coordinates inspected')

2026-11-02 21:50 INFO Native embedding fit not started; baseline coordinates inspected


Native V4 embedding fitting is a separate pipeline stage. Record the resolved feature-profile hash, selected features, seed, software versions, and input run before fitting.
